# imports

In [1]:
import numpy as np 
import tensorflow as tf 
import cv2 
import os 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report
import random
from tensorflow import keras
from tensorflow.keras.applications import EfficientNetB3
import kagglehub
from tensorflow.keras import layers
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras import regularizers

# read images 

In [2]:
path = kagglehub.dataset_download("pacificrm/skindiseasedataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/pacificrm/skindiseasedataset


In [3]:
Base_dir = os.path.join(path, 'SkinDisease', 'SkinDisease')
train_dir = os.path.join(Base_dir , 'train')
test_dir = os.path.join(Base_dir , 'test')

In [4]:
train_dirs = {}

for d in os.listdir(train_dir):
    d_path = os.path.join(train_dir, d)
    if os.path.isdir(d_path):
        train_dirs[d] = d_path

# CV2 (raed , resize)

In [5]:
image_size = (300,300)
Batch_size = 32

def load_img_labels(directory ,label):
    images = [] 
    labels = []
    image_file = [file for file in os.listdir(directory) if file.lower().endswith(('.png' ,'.jpg','.jpeg'))]
    for filename in image_file:
        filepath = os.path.join(directory , filename)
        image = cv2.imread(filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image ,image_size)
        images.append(image)
        labels.append(label)
    return images ,labels

# loading images and shuffle

In [6]:
classes = os.listdir(train_dir)
classes.remove("Lupus")
classes.remove('Infestations_Bites')

classlabel = {
    classname :i
    for i , classname in enumerate(classes)
}
print(classlabel)

{'Benign_tumors': 0, 'Unknown_Normal': 1, 'Tinea': 2, 'Eczema': 3, 'Actinic_Keratosis': 4, 'Vascular_Tumors': 5, 'Acne': 6, 'Rosacea': 7, 'Seborrh_Keratoses': 8, 'Moles': 9, 'Vitiligo': 10, 'SkinCancer': 11, 'Vasculitis': 12, 'Lichen': 13, 'Candidiasis': 14, 'DrugEruption': 15, 'Sun_Sunlight_Damage': 16, 'Bullous': 17, 'Warts': 18, 'Psoriasis': 19}


In [7]:
x_train = []
y_train =[]

for classname in classes:
    class_dir = os.path.join(train_dir ,classname)
    label = classlabel[classname]

    images ,labels = load_img_labels(class_dir , label)

    x_train.extend(images)
    y_train.extend(labels)

x_test = []
y_test= []

for classname in classes:
    class_dir = os.path.join(test_dir, classname)
    label = classlabel[classname]

    images, labels = load_img_labels(class_dir, label)

    x_test.extend(images)
    y_test.extend(labels)

In [8]:
x_train = np.array(x_train)
y_train = np.array(y_train)
x_test = np.array(x_test)
y_test = np.array(y_test)

In [9]:
def shuffle_data(images, labels):
    combined = list(zip(images, labels))
    random.shuffle(combined)
    shuffled_images, shuffled_labels = zip(*combined)
    return np.array(shuffled_images), np.array(shuffled_labels)

# Normalized

In [10]:
x_train, y_train = shuffle_data(x_train, y_train)
x_test, y_test = shuffle_data(x_test, y_test)

print(f"Total training images: {len(x_train)}")
print(f"Total testing images: {len(x_test)}")

Total training images: 13063
Total testing images: 1452


# Train Model

In [11]:
base_model = EfficientNetB3(
    input_shape=(300, 300, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

I0000 00:00:1788572518.871161      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [12]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.15),
    keras.layers.RandomContrast(0.15),
    keras.layers.RandomTranslation(0.1, 0.1) ,
])

In [13]:
model = keras.Sequential([
    data_augmentation,
    layers.Lambda(preprocess_input),
    base_model,
    keras.layers.GlobalAveragePooling2D() ,
    layers.Dense(128,activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    keras.layers.Flatten(),
    layers.Dense(20, activation='softmax')
])

In [14]:
base_model.trainable = True
for layer in base_model.layers[:-100]:
    layer.trainable = False

In [15]:
for layer in base_model.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

In [16]:
model.compile(
    optimizer = keras.optimizers.Nadam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [17]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [18]:
model.fit(x_train,y_train,batch_size=Batch_size,epochs=20,validation_split = 0.2 , callbacks=[early_stop])

Epoch 1/20


E0000 00:00:1788572550.977529      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1_1/efficientnetb3_1/block1b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


327/327 ━━━━━━━━━━━━━━━━━━━━ 108s 247ms/step - accuracy: 0.3374 - loss: 2.2801 - val_accuracy: 0.4853 - val_loss: 1.6812
Epoch 2/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 224ms/step - accuracy: 0.4918 - loss: 1.6598 - val_accuracy: 0.5434 - val_loss: 1.4844
Epoch 3/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 224ms/step - accuracy: 0.5568 - loss: 1.4198 - val_accuracy: 0.5890 - val_loss: 1.3321
Epoch 4/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 224ms/step - accuracy: 0.6192 - loss: 1.2206 - val_accuracy: 0.5989 - val_loss: 1.2900
Epoch 5/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 224ms/step - accuracy: 0.6706 - loss: 1.0544 - val_accuracy: 0.6265 - val_loss: 1.2869
Epoch 6/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 224ms/step - accuracy: 0.7099 - loss: 0.9193 - val_accuracy: 0.6544 - val_loss: 1.1496
Epoch 7/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 223ms/step - accuracy: 0.7539 - loss: 0.7963 - val_accuracy: 0.6506 - val_loss: 1.1732
Epoch 8/20
327/327 ━━━━━━━━━━━━━━━━━━━━ 73s 224ms/step - accuracy: 0.7844 - loss: 0.6903 - va

In [19]:
y_train_pred = model.predict(x_train)
y_test_pred = model.predict(x_test)

y_train_pred = np.argmax(y_train_pred, axis=1)
y_test_pred = np.argmax(y_test_pred, axis=1)

print(classification_report(y_train, y_train_pred))
print(classification_report(y_test, y_test_pred))

409/409 ━━━━━━━━━━━━━━━━━━━━ 66s 153ms/step
46/46 ━━━━━━━━━━━━━━━━━━━━ 7s 156ms/step
              precision    recall  f1-score   support

           0       0.84      0.87      0.85      1093
           1       1.00      0.99      0.99      1651
           2       0.68      0.94      0.79       923
           3       0.83      0.89      0.86      1010
           4       0.96      0.86      0.91       748
           5       0.91      0.83      0.87       543
           6       0.88      0.93      0.91       593
           7       0.92      0.90      0.91       254
           8       0.89      0.86      0.88       455
           9       0.91      0.78      0.84       361
          10       0.97      0.96      0.97       714
          11       0.87      0.85      0.86       693
          12       0.84      0.84      0.84       461
          13       0.90      0.77      0.83       553
          14       0.92      0.80      0.86       248
          15       0.91      0.82      0.86       

# Save Model

In [20]:
model.save("Skin_Diseases.keras")

In [21]:
model.save_weights("Skin_Diseases_weights.weights.h5")